In [11]:
import pandas as pd

In [12]:

df_accounts = pd.read_csv("../data/processed/accounts_clean.csv")
df_subscriptions = pd.read_csv("../data/processed/subscriptions_clean.csv")
df_feature_usage = pd.read_csv("../data/processed/feature_usage_clean.csv")
df_support = pd.read_csv("../data/processed/support_tickets_clean.csv")
df_churn = pd.read_csv("../data/processed/churn_events_clean.csv")

In [13]:
df_accounts['signup_date'] = pd.to_datetime(df_accounts['signup_date'])
df_subscriptions['start_date'] = pd.to_datetime(df_subscriptions['start_date'])
df_subscriptions['end_date'] = pd.to_datetime(df_subscriptions['end_date'])
df_feature_usage['usage_date'] = pd.to_datetime(df_feature_usage['usage_date'])
df_support['submitted_at'] = pd.to_datetime(df_support['submitted_at'])
df_support['closed_at'] = pd.to_datetime(df_support['closed_at'])
df_churn['churn_date'] = pd.to_datetime(df_churn['churn_date'])

In [14]:
print("Processed data loaded, dates re-converted.")

Processed data loaded, dates re-converted.


In [15]:
reference_date = pd.Timestamp("2024-12-31")
df_accounts['tenure_days'] = (reference_date - df_accounts['signup_date']).dt.days

In [16]:
active_subs = df_subscriptions[df_subscriptions['is_active'] == True]
active_mrr = active_subs.groupby('account_id')['mrr_amount'].sum().reset_index()
active_mrr.columns = ['account_id', 'current_mrr']

df_accounts = df_accounts.merge(active_mrr, on='account_id', how='left')
df_accounts['current_mrr'] = df_accounts['current_mrr'].fillna(0)

In [17]:
usage_per_sub = df_feature_usage.groupby('subscription_id')['usage_count'].sum().reset_index()
usage_with_account = usage_per_sub.merge(df_subscriptions[['subscription_id', 'account_id']], on='subscription_id', how='left')
usage_per_account = usage_with_account.groupby('account_id')['usage_count'].sum().reset_index()
usage_per_account.columns = ['account_id', 'total_usage_count']

df_accounts = df_accounts.merge(usage_per_account, on='account_id', how='left')
df_accounts['total_usage_count'] = df_accounts['total_usage_count'].fillna(0)

In [18]:
df_accounts[['account_id', 'tenure_days', 'current_mrr', 'total_usage_count']].head()

,account_id,tenure_days,current_mrr,total_usage_count
0,A-2e4581,76,12603,535
1,A-43a9e3,502,10004,355
2,A-0a282f,126,13311,821
3,A-1f0ac7,492,9275,382
4,A-ce550d,65,25355,579


In [19]:
support_agg = df_support.groupby('account_id').agg(
    ticket_count=('ticket_id', 'count'),
    avg_satisfaction=('satisfaction_score', 'mean'),
    escalation_rate=('escalation_flag', 'mean')
).reset_index()

df_accounts = df_accounts.merge(support_agg, on='account_id', how='left')

# Accounts with zero tickets won't appear in df_support at all - fill with 0, not NaN
df_accounts['ticket_count'] = df_accounts['ticket_count'].fillna(0)
df_accounts['escalation_rate'] = df_accounts['escalation_rate'].fillna(0)
# avg_satisfaction stays NaN if no tickets exist - there's genuinely no satisfaction score to report

In [20]:
df_accounts[['account_id', 'ticket_count', 'avg_satisfaction', 'escalation_rate']].head(10)

,account_id,ticket_count,avg_satisfaction,escalation_rate
0,A-2e4581,2.0,3.000000,0.000000
1,A-43a9e3,3.0,4.000000,0.000000
2,A-0a282f,3.0,4.666667,0.000000
3,A-1f0ac7,2.0,NaN,0.000000
4,A-ce550d,7.0,3.800000,0.142857
5,A-1b9609,4.0,3.000000,0.000000
6,A-a0ca4e,6.0,3.800000,0.500000
7,A-e5d6ab,3.0,NaN,0.000000
8,A-7dacce,4.0,3.666667,0.000000
9,A-10b8da,3.0,3.500000,0.000000


In [21]:
behavior_flags = df_subscriptions.groupby('account_id').agg(
    has_upgraded=('upgrade_flag', 'max'),
    has_downgraded=('downgrade_flag', 'max')
).reset_index()

df_accounts = df_accounts.merge(behavior_flags, on='account_id', how='left')

In [22]:
df_accounts.columns.tolist()

['account_id',
 'account_name',
 'industry',
 'country',
 'signup_date',
 'referral_source',
 'plan_tier',
 'seats',
 'is_trial',
 'churn_flag',
 'tenure_days',
 'current_mrr',
 'total_usage_count',
 'ticket_count',
 'avg_satisfaction',
 'escalation_rate',
 'has_upgraded',
 'has_downgraded']

In [23]:
df_accounts.isnull().sum()

account_id            0
account_name          0
industry              0
country               0
signup_date           0
referral_source       0
plan_tier             0
seats                 0
is_trial              0
churn_flag            0
tenure_days           0
current_mrr           0
total_usage_count     0
ticket_count          0
avg_satisfaction     34
escalation_rate       0
has_upgraded          0
has_downgraded        0
dtype: int64

In [24]:
df_accounts.to_csv("../data/processed/account_features.csv", index=False)
print("Feature table saved.")

Feature table saved.
